In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tb_macro.constants import DATA_PATH

plt.style.use("ggplot")

iso3s = ["BRA", "RUS", "IND", "CHN", "ZAF"]
labels = {
    "BRA": "Brazil",
    "RUS": "Russia",
    "IND": "India",
    "CHN": "China",
    "ZAF": "South Africa",
}

# Inputs
who_data = pd.read_csv(DATA_PATH / "who/who_indicators_20261007T0317Z.csv")
who_data = who_data[who_data["iso3"].isin(iso3s)].sort_values(["iso3", "year"])

wdi = pd.read_csv(DATA_PATH / "worldbank/wdi_20261007T0317Z.csv")
wdi = wdi[wdi["iso3"].isin(iso3s)].sort_values(["iso3", "year"])
gdp = wdi[wdi["indicator_id"] == "NY.GDP.PCAP.KD"]
undernut = wdi[wdi["indicator_id"] == "SN.ITK.DEFC.ZS"]

# Collation
panels = [
    (who_data, "e_inc_100k", "TB incidence (per 100,000)"),
    (who_data, "e_mort_exc_tbhiv_100k", "TB mortality (per 100,000)"),
    (gdp, "value", "GDP per capita (2015 USD)"),
    (undernut, "value", "undernourishment (percent)"),
]

# Plotting
fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
for ax, (data, col, ylabel) in zip(axes.ravel(), panels):
    ax.set_ylabel(ylabel)
    for iso3 in iso3s:
        c = data[data["iso3"] == iso3].dropna(subset=[col])
        ax.plot(c["year"], c[col], label=labels[iso3])

# Cosmetics
for a, ax in enumerate(axes.ravel()):
    ax.set_yscale("log")
axes[0, 1].set_ylim(bottom=1)
axes[1, 0].set_ylim(bottom=500)
axes[1, 1].set_ylim(bottom=0.9, top=25)
axes[0, 0].legend(loc="upper right")
for ax in axes[1, :]:
    ax.set_xlim(2000, 2020)
    ax.set_xticks(np.arange(2000, 2025, 5))

fig.tight_layout()
fig.savefig("figure_6_ft27.pdf")